In [1]:
import os, sys, dill, yaml , json
import numpy as np
from pandas import DataFrame
import pandas as pd

In [2]:
def write_yaml_file(filepath: str, content: object, replace: bool = True) -> None:
    
    """
    Writes content to a YAML file. Optionally prevents overwriting unless 'replace=True'.

    Parameters:
        filepath (str): Path to the YAML file.
        content (object): Python object to serialize as YAML.
        replace (bool): If False and file exists, raises an error. Default is False.

    Raises:
        USvisaException: If any error occurs during file operation.
    """
    
    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    with open(filepath, "w") as file:
        yaml.dump(content, file)

In [3]:
from evidently import Report
from evidently.presets import DataSummaryPreset , DataDriftPreset 

In [4]:
df=pd.read_csv("Visadataset.csv" )
df.head()

,case_id,continent,education_of_employee,has_job_experience,requires_job_training,no_of_employees,yr_of_estab,region_of_employment,prevailing_wage,unit_of_wage,full_time_position,case_status
0,EZYV01,Asia,High School,N,N,14513,2007,West,592.2029,Hour,Y,Denied
1,EZYV02,Asia,Master's,Y,N,2412,2002,Northeast,83425.6500,Year,Y,Certified
2,EZYV03,Asia,Bachelor's,N,Y,44444,2008,West,122996.8600,Year,Y,Denied
3,EZYV04,Asia,Bachelor's,N,N,98,1897,West,83434.0300,Year,Y,Denied
4,EZYV05,Africa,Master's,Y,N,1082,2005,South,149907.3900,Year,Y,Certified


In [5]:
from sklearn.model_selection import train_test_split

In [6]:
train_df, test_df = train_test_split(df, test_size=0.3)

In [7]:
def detect_dataset_drift(
                        reference_df:DataFrame,
                        current_df:DataFrame
                        ) ->bool :
        
    """
    This Method validates if drift is detected.
    
    Returns : returns bool value based on validation results.
    """
    
    evidently_ai_report_obj = Report( [DataDriftPreset()], include_tests="True" )
    evidently_report_snapshort_obj = evidently_ai_report_obj.run(reference_data=reference_df, current_data=current_df)
    
    print("evidently_report_snapshort_obj : " ,evidently_report_snapshort_obj)
    print("type evidently_report_snapshort_obj : " , type(evidently_report_snapshort_obj))
    
    evidently_report_snapshort_obj.save_html("stats_file.html")

    # returns the json in str format
    report_str = evidently_report_snapshort_obj.json()        # need to change this method
    
    # Deserialize  (str, bytes or byte array instance containing a JSON document) to a Python object.
    json_report=json.loads(report_str)           
    
    write_yaml_file(filepath="data_drift/data_drift_report.yaml",
                    content=json_report)
  
    return json_report

In [8]:
json_report = detect_dataset_drift(reference_df=train_df, current_df=test_df)

evidently_report_snapshort_obj :  <evidently.core.report.Snapshot object at 0x000002E96F988BE0>
type evidently_report_snapshort_obj :  <class 'evidently.core.report.Snapshot'>


In [9]:
json_report

{'metrics': [{'id': '15e89f895b482f9b84ba7274ed18a106',
   'metric_id': 'DriftedColumnsCount(drift_share=0.5)',
   'value': {'count': 0.0, 'share': 0.0}},
  {'id': '44ed95692690fdd0122a65b138e9060b',
   'metric_id': 'ValueDrift(column=no_of_employees)',
   'value': 0.008018277294367132},
  {'id': 'a1076b7f552ed954d7a0f7b5e3389aed',
   'metric_id': 'ValueDrift(column=yr_of_estab)',
   'value': 0.010770938240899034},
  {'id': '3f76acd90ddbe278fb63f5ef3169f600',
   'metric_id': 'ValueDrift(column=prevailing_wage)',
   'value': 0.00943771849449775},
  {'id': '331f8786a2cc5d42160cb3239d26369e',
   'metric_id': 'ValueDrift(column=continent)',
   'value': 0.008831991215659545},
  {'id': '462f7e3097be07eeb3f21ae6d3142f81',
   'metric_id': 'ValueDrift(column=education_of_employee)',
   'value': 0.007414197816680817},
  {'id': '7f345702abe4aaa71a56794154694dd5',
   'metric_id': 'ValueDrift(column=has_job_experience)',
   'value': 0.0020626950492821554},
  {'id': 'cfd219a499d8abe9685c9fea4f9d0abe

In [10]:
overall_drift_Status= {}
columnwise_drift_report = {}

for obj in json_report['tests'] :

    if obj['id'] == 'lt':

        overall_drift_Status["name"] = obj['name']
        overall_drift_Status["description"] = obj['description']
        overall_drift_Status["status"] = obj['status']
        overall_drift_Status["threshold"] = obj['bound_test']['test']['threshold']

    else : 

        columnwise_drift_report[obj['metric_config']['params']['column']]   = [{"description" : obj['description'] , 
                                                                               "status" : obj['status']
                                                                               }]
        #columnwise_drift_report[obj['metric_config']['params']['column']].append({obj['description']})
        

In [26]:
overall_drift_Status , columnwise_drift_report

({'name': 'Share of Drifted Columns: Less 0.500',
  'description': 'Share of Drifted Columns: Actual value 0.000 < 0.500',
  'status': 'SUCCESS',
  'threshold': 0.5},
 {'no_of_employees': [{'description': 'Drift score is 0.02. The drift detection method is Wasserstein distance (normed). The drift threshold is 0.10.',
    'status': 'SUCCESS'}],
  'yr_of_estab': [{'description': 'Drift score is 0.02. The drift detection method is Wasserstein distance (normed). The drift threshold is 0.10.',
    'status': 'SUCCESS'}],
  'prevailing_wage': [{'description': 'Drift score is 0.03. The drift detection method is Wasserstein distance (normed). The drift threshold is 0.10.',
    'status': 'SUCCESS'}],
  'continent': [{'description': 'Drift score is 0.01. The drift detection method is Jensen-Shannon distance. The drift threshold is 0.10.',
    'status': 'SUCCESS'}],
  'education_of_employee': [{'description': 'Drift score is 0.01. The drift detection method is Jensen-Shannon distance. The drift th

In [11]:
# Save as JSON file
with open('overall_drift_status.json', 'w') as f:
    json.dump(overall_drift_Status, f, indent=4)

In [12]:
# Convert dictionary to DataFrame
df = pd.DataFrame.from_dict({k: v[0] for k, v in columnwise_drift_report.items()}, orient='index')

# Optional: Rename index name
df.index.name = 'feature'

# Show DataFrame
df.to_csv("Columnwise_drift_report.csv")